In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score,roc_curve,RocCurveDisplay
from sklearn.metrics import recall_score, precision_score, classification_report,log_loss
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder,StandardScaler,LabelEncoder
from sklearn.compose import  ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_curve, RocCurveDisplay, roc_auc_score
import os
os.chdir(r'../datasets')
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [28]:
imgseg = pd.read_csv('Image_Segmentation.csv')
imgseg

,Class,region.centroid.col,region.centroid.row,region.pixel.count,short.line.density.5,short.line.density.2,vedge.mean,vegde.sd,hedge.mean,hedge.sd,intensity.mean,rawred.mean,rawblue.mean,rawgreen.mean,exred.mean,exblue.mean,exgreen.mean,value.mean,saturation.mean,hue-mean
0,BRICKFACE,188,133,9,0.000000,0.0,0.333333,0.266667,0.500000,0.077778,6.666666,8.333334,7.777778,3.888889,5.000000,3.333333,-8.333333,8.444445,0.538580,-0.924817
1,BRICKFACE,105,139,9,0.000000,0.0,0.277778,0.107407,0.833333,0.522222,6.111111,7.555555,7.222222,3.555556,4.333334,3.333333,-7.666666,7.555555,0.532628,-0.965946
2,BRICKFACE,34,137,9,0.000000,0.0,0.500000,0.166667,1.111111,0.474074,5.851852,7.777778,6.444445,3.333333,5.777778,1.777778,-7.555555,7.777778,0.573633,-0.744272
3,BRICKFACE,39,111,9,0.000000,0.0,0.722222,0.374074,0.888889,0.429629,6.037037,7.000000,7.666666,3.444444,2.888889,4.888889,-7.777778,7.888889,0.562919,-1.175773
4,BRICKFACE,16,128,9,0.000000,0.0,0.500000,0.077778,0.666667,0.311111,5.555555,6.888889,6.666666,3.111111,4.000000,3.333333,-7.333334,7.111111,0.561508,-0.985811
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,GRASS,36,243,9,0.111111,0.0,1.888889,1.851851,2.000000,0.711110,13.333333,9.888889,12.111111,18.000000,-10.333333,-3.666667,14.000000,18.000000,0.452229,2.368311
205,GRASS,186,218,9,0.000000,0.0,1.166667,0.744444,1.166667,0.655555,13.703704,10.666667,12.666667,17.777779,-9.111111,-3.111111,12.222222,17.777779,0.401347,2.382684
206,GRASS,197,236,9,0.000000,0.0,2.444444,6.829628,3.333333,7.599998,16.074074,13.111111,16.666668,18.444445,-8.888889,1.777778,7.111111,18.555555,0.292729,2.789800
207,GRASS,208,240,9,0.111111,0.0,1.055556,0.862963,2.444444,5.007407,14.148149,10.888889,13.000000,18.555555,-9.777778,-3.444444,13.222222,18.555555,0.421621,2.392487


In [29]:
# 1. Reload the data to "reset" the Class column
imgseg = pd.read_csv('Image_Segmentation.csv')

# 2. Initialize
le = LabelEncoder()
lr = LogisticRegression(max_iter=1000)

# 3. Transform Class and define features
imgseg['Class'] = le.fit_transform(imgseg['Class'])
X = imgseg.drop("Class", axis=1)
y = imgseg['Class']

# 4. Split with stratification (ensures all classes are represented)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=26, stratify=y
)

# 5. Fit the modelsit correct
lr.fit(X_train, y_train)

# 6. Predict and calculate Log Loss
y_pred_prob = lr.predict_proba(X_test)
print("Log Loss:", log_loss(y_test, y_pred_prob))

Log Loss: 1.1454431795724527


In [30]:
solvers = ["lbfgs","newton-cg", "newton-cholesky", "sag", "saga"]
Cs = np.linspace(0.001, 5, 20)
scores = []

for s in solvers:
    for c in Cs:
        lr = LogisticRegression(solver=s, C=c, max_iter=1000) 
        lr.fit(X_train, y_train)
        y_pred = lr.predict(X_test)
        f1 = f1_score(y_test, y_pred, average = 'macro')
        scores.append([s, c, f1])
df_scores = pd.DataFrame(scores, columns=['solver', 'C', 'score'])
df_scores.sort_values('score', ascending=True)

,solver,C,score
60,sag,0.001000,0.766596
80,saga,0.001000,0.766596
19,lbfgs,5.000000,0.852945
61,sag,0.264105,0.853795
63,sag,0.790316,0.853795
...,...,...,...
35,newton-cg,3.947579,0.924826
36,newton-cg,4.210684,0.924826
39,newton-cg,5.000000,0.924826
41,newton-cholesky,0.264105,0.939362
